<a href="https://colab.research.google.com/github/jhuangjennifer/573ChineseEnglishSummarization/blob/jjnhuang-work/notebooks/inference_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -U transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 140.5 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [2]:
# Load summarization model directly
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

bart_tokenizer = AutoTokenizer.from_pretrained("yunu919/bart-large-dialogue-summarization")
bart_model = AutoModelForSeq2SeqLM.from_pretrained("yunu919/bart-large-dialogue-summarization")

mbart_tokenizer = AutoTokenizer.from_pretrained("yunu919/mbart-large-dialogue-summarization")
mbart_model = AutoModelForSeq2SeqLM.from_pretrained("yunu919/mbart-large-dialogue-summarization")

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/360 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/813M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.44G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/516 [00:00<?, ?it/s]

generation_config.json: 0.00B [00:00, ?B/s]

In [3]:
# Initialize translation tokenizer and model: https://huggingface.co/Helsinki-NLP/opus-mt-en-zhhttps://huggingface.co/Helsinki-NLP/opus-mt-en-zh

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

translate_tokenizer = AutoTokenizer.from_pretrained("Helsinki-NLP/opus-mt-en-zh")
translate_model = AutoModelForSeq2SeqLM.from_pretrained("Helsinki-NLP/opus-mt-en-zh")

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/44.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/806k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/805k [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/models/marian/tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


pytorch_model.bin:   0%|          | 0.00/312M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

In [5]:
# Load XSAMSum dataset
from datasets import load_dataset
import os

BASE_DIR = f"sample_data"

data_files = {
    "test": os.path.join(BASE_DIR, "test.json"),
}

dataset_dict = load_dataset("json", data_files=data_files)

Generating test split: 0 examples [00:00, ? examples/s]

In [9]:
def summarize_dialogue(text, summary_tokenizer, summary_model):
    # 1. Prepare input - this returns a dict containing 'input_ids' and 'attention_mask'
    inputs = summary_tokenizer(text, return_tensors="pt", max_length=512, truncation=True).to(summary_model.device)

    # 2. Generate summary - explicitly pass both input_ids and attention_mask
    summary_ids = summary_model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"], # Crucial for MPS compatibility
        max_length=150,
        num_beams=4,
        early_stopping=True
    )

    # 3. Decode summary
    summary = summary_tokenizer.decode(summary_ids[0], skip_special_tokens=True)

    # 4. Translate summary
    tokenized_summary = translate_tokenizer(summary, return_tensors="pt").to(translate_model.device)
    output = translate_model.generate(**tokenized_summary)

    # 5. Decode and return translated summary
    decoded_output = translate_tokenizer.decode(output[0], skip_special_tokens=True)
    return decoded_output

# Set this to BART or mBART
tokenizer = mbart_tokenizer # Either bart_tokenizer or mbart_tokenizer
model = mbart_model # Either bart_model or mbart_model

output_filepath = os.path.join(BASE_DIR, "output/predictions.txt")
with open(output_filepath, "w") as f:
  for i in range(len(dataset_dict['test']['dialogue'])):
    text = dataset_dict['test']['dialogue'][i]
    prediction = summarize_dialogue(text, tokenizer, model)
    f.write(prediction + "\n")
    if (i % 100) == 0:
      print(str(i+1) + " sample(s) summarized")
    elif (i == (len(dataset_dict['test']['dialogue'])-1)):
      print("All " + str(i+1) + "samples summarized")

1 sample(s) summarized


KeyboardInterrupt: 

In [12]:
from google.colab import output
output.clear()